# 02. Будущие прогнозные интерфейсы — без обучения и прогнозов

TASK R1.4, §7. «Розетка» означает место подключения сменного оценивателя, а не уже работающую нейросеть. Здесь **нет численных предсказаний, весов, обучения и оценки точности**. Старые результаты исторической тетради не используются.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
root = Path.cwd()
while root != root.parent and not (root/'efc/companion.py').exists():
    root = root.parent
if not (root/'efc/companion.py').exists():
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('Запустите из копии night/fefc-v3 или в Colab.')
    root = Path('/content/fefc-r14-companions')
    if not root.exists():
        subprocess.run(['git','clone','--single-branch','--branch','night/fefc-v3',
            'https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(root)],check=True)
    if not (root/'efc/companion.py').exists():
        raise RuntimeError('Копия устарела: companion.py отсутствует. Молчаливого обновления нет.')
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements.txt')],check=True)
os.chdir(root)
sys.path.insert(0,str(root))
SOURCE_SHA = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip() if (root/'.git').exists() else os.environ.get('FEFC_SOURCE_SHA','архив: SHA в реестре')
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib','inline')
import matplotlib.pyplot as plt
import pandas as pd
from efc import companion as c
print('Источник:',SOURCE_SHA,'; Python:',sys.version.split()[0])
print('Только чтение и схемы: имитация, обучение, прогноз и применение плана не запускаются.')

## 1. Три разных контракта

Оценка p95/потерь конфигурации, риск переполнения и риск отказа датчика требуют разных входов. MLP, ближайшие соседи и M/M/1/K — возможные реализации первого порта в будущем; их сравнение сейчас не выполнено. `None` означает отсутствие оценки, а не нулевую вероятность.

In [ ]:
contracts = c.port_contracts()
assert all(p['estimate'] is None and p['status']=='NOT_IMPLEMENTED' for p in contracts)
display(pd.DataFrame(contracts))

## 2. Место подключения

Выход должен сопровождаться версией, единицами, областью применимости и отдельно проверенной калибровкой. Схема ниже описывает будущую передачу данных; стрелки не означают действующее вычисление.

In [ ]:
fig = c.concept_figure('ports')
display(fig); plt.close(fig)

## 3. Что нельзя подменять

Модельные p95 и потери не являются выходами нейросети. Прогноз p95 конфигурации не создаёт автоматически сопоставимых порядковых команд для целых подграфов. Для такой связи нужен отдельный адаптер и проверка области применимости; он не реализован.

Порядковая команда не измеряется в секундах, байтах или вероятностях. Горизонт упреждения не является гарантированным резервом времени. Ручное отключение датчика в 00 не является прогнозом или диагностикой отказа; отсутствие сообщения само по себе отказ не устанавливает. Данных для обучения риска отказов здесь нет.

Обработчик `invoke_port` намеренно отклоняет вызов: исторический файл весов не включает порт автоматически. Проверка этого запрета находится в `tests/test_companion.py`.

In [ ]:
print(json.dumps(dict(notebook='02',source=SOURCE_SHA,ports=[p['id'] for p in contracts],numeric_outputs=None,forecast_executed=False,training_executed=False,controller_executed=False,colab_browser_checked=False),ensure_ascii=False,indent=2))